# Import


In [8]:
import shutil
import torch
import tempfile
from pathlib import Path
from mrpro.data import KData  # Import the KData class
from mrpro.data.traj_calculators import KTrajectoryPulseq
import numpy as np
import requests
import mrpro
import matplotlib.pyplot as plt

# Use the trajectory that is stored in the ISMRMRD file
# trajectory = mrpro.data.traj_calculators.KTrajectoryIsmrmrd(seq_file)
# Load in the Data from the ISMRMRD file
# Load data from Pulseq file using KTrajectoryPulseq
# Local path
h5_path = '/home/bouill01/data/20240319_spiral_2D_256mm_220k0_128interleaves_golden_angle_vds/pulseq_spiral_2D_220k0_128interleaves_golden_angle_vds_with_traj.h5'
seq_path = '/home/bouill01/data/20240319_spiral_2D_256mm_220k0_128interleaves_golden_angle_vds/20240319_spiral_2D_256mm_220k0_128interleaves_golden_angle_vds.seq'
kdatapuls = KData.from_file(h5_path, KTrajectoryPulseq(seq_path=seq_path))

## Def shift

In [9]:
from mrpro.data import KData, KTrajectory
from mrpro.data.traj_calculators import KTrajectoryPulseq
import matplotlib.pyplot as plt
import numpy as np

def shift_k_space_trajectory(kdatapuls):
    # Extract k-space trajectory from kdatapuls
    ky_pulseq = kdatapuls.traj.ky
    kx_pulseq = kdatapuls.traj.kx
    kz_pulseq = kdatapuls.traj.kz

    # Number of indices
    num_indices = ky_pulseq.shape[2]

    # Initialize lists to store shifted trajectories
    shifted_ky = ky_pulseq.clone()
    shifted_kx = kx_pulseq.clone()

    # Loop to apply the shift to each index
    for i in range(num_indices-1):
        # Calculate the shift for the current index
        shifted_ky[:,:,i,:] -= ky_pulseq[:,:,i,0]
        shifted_kx[:,:,i,:] -= kx_pulseq[:,:,i,0]

    # Create shifted KTrajectory object
    shifted_traj = KTrajectory(kx=shifted_kx, ky=shifted_ky, kz=kz_pulseq)
    # Create shifted KData object
    shifted_kdatapuls = KData(data=kdatapuls.data, traj=shifted_traj, header=kdatapuls.header)

    return shifted_kdatapuls

In [10]:
shifted_kdatapuls = shift_k_space_trajectory(kdatapuls)

## Unet

In [11]:
import torch
import torch.nn as nn
import numpy as np


class ConvBlock(nn.Module):
    """
    A simple block of convolutional layers (1D, 2D or 3D)
    """

    def __init__(
        self,
        dim,
        n_ch_in,
        n_ch_out,
        n_convs,
        kernel_size=3,
        bias=True,
        padding_mode="zeros",
    ):
        super().__init__()

        if dim == 1:
            conv_op = nn.Conv1d
        if dim == 2:
            conv_op = nn.Conv2d
        elif dim == 3:
            conv_op = nn.Conv3d

        padding = int(np.floor(kernel_size / 2))

        conv_block_list = []
        conv_block_list.extend(
            [
                conv_op(
                    n_ch_in,
                    n_ch_out,
                    kernel_size,
                    padding=padding,
                    bias=bias,
                    padding_mode=padding_mode,
                ),
                nn.LeakyReLU(),
            ]
        )

        for i in range(n_convs - 1):
            conv_block_list.extend(
                [
                    conv_op(
                        n_ch_out,
                        n_ch_out,
                        kernel_size,
                        padding=padding,
                        bias=bias,
                        padding_mode=padding_mode,
                    ),
                    nn.LeakyReLU(),
                ]
            )

        self.conv_block = nn.Sequential(*conv_block_list)

    def forward(self, x):
        return self.conv_block(x)


class Encoder(nn.Module):
    def __init__(
        self,
        dim,
        n_ch_in,
        n_enc_stages,
        n_convs_per_stage,
        n_filters,
        kernel_size=3,
        bias=True,
        padding_mode="zeros",
    ):
        super().__init__()

        n_ch_list = [n_ch_in]
        for ne in range(n_enc_stages):
            n_ch_list.append(int(n_filters) * 2**ne)

        self.enc_blocks = nn.ModuleList(
            [
                ConvBlock(
                    dim,
                    n_ch_list[i],
                    n_ch_list[i + 1],
                    n_convs_per_stage,
                    kernel_size=kernel_size,
                    bias=bias,
                    padding_mode=padding_mode,
                )
                for i in range(len(n_ch_list) - 1)
            ]
        )

        if dim == 1:
            pool_op = nn.MaxPool1d(2)
        elif dim == 2:
            pool_op = nn.MaxPool2d(2)
        elif dim == 3:
            pool_op = nn.MaxPool3d(2)

        self.pool = pool_op

    def forward(self, x):
        features = []
        for block in self.enc_blocks:
            x = block(x)
            features.append(x)
            x = self.pool(x)
        return features


class Decoder(nn.Module):
    def __init__(
        self,
        dim,
        n_ch_in,
        n_dec_stages,
        n_convs_per_stage,
        kernel_size=3,
        bias=False,
        padding_mode="zeros",
    ):
        super().__init__()

        n_ch_list = []
        for ne in range(n_dec_stages):
            n_ch_list.append(int(n_ch_in * (1 / 2) ** ne))

        if dim == 1:
            conv_op = nn.Conv1d
            interp_mode = "linear"
        elif dim == 2:
            conv_op = nn.Conv2d
            interp_mode = "bilinear"
        elif dim == 3:
            interp_mode = "trilinear"
            conv_op = nn.Conv3d

        self.interp_mode = interp_mode

        padding = int(np.floor(kernel_size / 2))
        self.upconvs = nn.ModuleList(
            [
                conv_op(
                    n_ch_list[i],
                    n_ch_list[i + 1],
                    kernel_size=kernel_size,
                    padding=padding,
                    bias=bias,
                    padding_mode=padding_mode,
                )
                for i in range(len(n_ch_list) - 1)
            ]
        )
        self.dec_blocks = nn.ModuleList(
            [
                ConvBlock(
                    dim,
                    n_ch_list[i],
                    n_ch_list[i + 1],
                    n_convs_per_stage,
                    kernel_size=kernel_size,
                    bias=bias,
                    padding_mode=padding_mode,
                )
                for i in range(len(n_ch_list) - 1)
            ]
        )

    def forward(self, x, encoder_features):
        for i in range(len(self.dec_blocks)):
            enc_features = encoder_features[i]
            enc_features_shape = enc_features.shape
            x = nn.functional.interpolate(
                x, enc_features_shape[2:], mode=self.interp_mode, align_corners=False
            )
            x = self.upconvs[i](x)
            x = torch.cat([x, enc_features], dim=1)
            x = self.dec_blocks[i](x)
        return x


class UNet(nn.Module):
    def __init__(
        self,
        dim,
        n_ch_in=2,
        n_ch_out=2,
        n_enc_stages=3,
        n_convs_per_stage=2,
        n_filters=16,
        kernel_size=3,
        bias=True,
        residual_connection=True,
        padding_mode="zeros",
    ):
        super().__init__()
        self.encoder = Encoder(
            dim,
            n_ch_in,
            n_enc_stages,
            n_convs_per_stage,
            n_filters,
            kernel_size=kernel_size,
            bias=bias,
            padding_mode=padding_mode,
        )
        self.decoder = Decoder(
            dim,
            n_filters * (2 ** (n_enc_stages - 1)),
            n_enc_stages,
            n_convs_per_stage,
            kernel_size=kernel_size,
            bias=bias,
            padding_mode=padding_mode,
        )

        if dim == 1:
            conv_op = nn.Conv1d
        elif dim == 2:
            conv_op = nn.Conv2d
        elif dim == 3:
            conv_op = nn.Conv3d

        self.c1x1 = conv_op(n_filters, n_ch_out, kernel_size=1, padding=0, bias=bias)

        self.residual_connection = residual_connection
        if residual_connection:
            if n_ch_in != n_ch_out:
                raise ValueError(
                    "For using the residual connection, the number of input and output channels of the \
                    network must be the same.\
                    Given {n_ch_in} and {n_ch_out}."
                )

    def forward(self, x):
        enc_features = self.encoder(x)
        dec = self.decoder(enc_features[-1], enc_features[::-1][1:])
        out = self.c1x1(dec)
        if self.residual_connection:
            out = out + x
        return out

## Nufft

In [12]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader
import numpy as np

class NUFFTCascade(nn.Module):
    def __init__(self, acquisition_operator, unet, nu, npcg, w, learn_lambda=True):
        super(NUFFTCascade, self).__init__()
        self.acquisition_operator = acquisition_operator
        self.unet = unet
        self.nu = nu
        self.npcg = npcg
        self.w = w
        self.lambda_param = nn.Parameter(torch.tensor(0.1), requires_grad=learn_lambda)

    def forward(self, x, k_space_data):
        for _ in range(self.npcg):
            operator_test = self.acquisition_operator.H(self.acquisition_operator(x) - k_space_data)
            grad_unet_x = torch.autograd.grad(outputs=self.unet(x), inputs=x, grad_outputs=torch.ones_like(self.unet(x)), retain_graph=True)[0]
            x = x - self.w * operator_test - self.w * grad_unet_x
        return x

## Ellipsoide

In [13]:
"""Numerical phantom with ellipses."""

# Copyright 2023 Physikalisch-Technische Bundesanstalt
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at:
#
#       http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

from collections.abc import Sequence

import numpy as np
import torch
from einops import repeat

from mrpro.data import SpatialDimension
from mrpro.phantoms.phantom_elements import EllipseParameters


class EllipsePhantom:
    """Numerical phantom as the sum of different ellipses.

    Parameters
    ----------
        ellipses
            ellipses defined by their center, radii and intensity.
            if None, defaults to three ellipses
    """

    def __init__(self, ellipses: Sequence[EllipseParameters] | None = None):
        """Initialize ellipse phantom.

        Parameters
        ----------
        ellipses
            Sequence of EllipseParameters defining the ellipses.
            if None, defaults to three ellipses with different parameters.
        """
        if ellipses is None:
            self.ellipses = [
                EllipseParameters(center_x=0.2, center_y=0.2, radius_x=0.1, radius_y=0.25, intensity=1),
                EllipseParameters(center_x=0.1, center_y=-0.1, radius_x=0.3, radius_y=0.1, intensity=2),
                EllipseParameters(center_x=-0.2, center_y=0.2, radius_x=0.18, radius_y=0.25, intensity=4),
            ]
        else:
            self.ellipses = list(ellipses)

    def kspace(self, ky: torch.Tensor, kx: torch.Tensor) -> torch.Tensor:
        """Create 2D analytic kspace data based on given k-space locations.

        For a corresponding image with 256 x 256 voxel, the k-space locations should be defined within [-128, 127]

        The Fourier representation of ellipses can be analytically described by Bessel functions. Further information
        and derivations can be found e.g. here: https://doi.org/10.1002/mrm.21292

        Parameters
        ----------
        ky
            k-space locations in ky
        kx
            k-space locations in kx (frequency encoding direction). Same shape as ky.
        """
        # kx and ky have to be of same shape
        if kx.shape != ky.shape:
            raise ValueError(f'shape mismatch between kx {kx.shape} and ky {ky.shape}')

        kdata = torch.zeros_like(kx, dtype=torch.complex64)
        for ellipse in self.ellipses:
            arg = torch.sqrt((ellipse.radius_x * 2) ** 2 * kx**2 + (ellipse.radius_y * 2) ** 2 * ky**2)
            arg[arg < 1e-6] = 1e-6  # avoid zeros

            cdata = 2 * 2 * ellipse.radius_x * ellipse.radius_y * 0.5 * torch.special.bessel_j1(torch.pi * arg) / arg
            kdata += (
                torch.exp(-1j * 2 * torch.pi * (ellipse.center_x * kx + ellipse.center_y * ky))
                * cdata
                * ellipse.intensity
            )

        # Scale k-space data by factor sqrt(number of points) to ensure correct scaling after FFT with
        # normalization "ortho". See e.g. https://docs.scipy.org/doc/scipy/tutorial/fft.html
        kdata *= np.sqrt(torch.numel(kdata))
        return kdata

    def image_space(self, image_dimensions: SpatialDimension[int]) -> torch.Tensor:
        """Create image representation of phantom.

        Parameters
        ----------
        image_dimensions
            number of voxels in the image
            This is a 2D simulation so the output will be (1 1 1 image_dimensions.y image_dimensions.x)
        """
        # Calculate image representation of phantom
        ny, nx = image_dimensions.y, image_dimensions.x
        ix, iy = torch.meshgrid(
            torch.linspace(-nx // 2, nx // 2 - 1, nx),
            torch.linspace(-ny // 2, ny // 2 - 1, ny),
            indexing='xy',
        )

        idata = torch.zeros((ny, nx), dtype=torch.complex64)
        for ellipse in self.ellipses:
            in_ellipse = (
                (ix / nx - ellipse.center_x) ** 2 / ellipse.radius_x**2
                + (iy / ny - ellipse.center_y) ** 2 / ellipse.radius_y**2
            ) <= 1
            idata += ellipse.intensity * in_ellipse

        return repeat(idata, 'y x->other coils z y x', other=1, coils=1, z=1)

In [14]:
import torch

# Assurez-vous que la classe EllipsePhantom et EllipseParameters sont définies ici
# ou importez-les depuis votre module

def generate_random_ellipses(num_ellipses):
    """Génère une liste de paramètres d'ellipses avec des valeurs aléatoires."""
    ellipses = []
    for _ in range(num_ellipses):
        center_x = np.random.uniform(-0.5, 0.5)
        center_y = np.random.uniform(-0.5, 0.5)
        radius_x = np.random.uniform(0.05, 0.3)
        radius_y = np.random.uniform(0.05, 0.3)
        intensity = np.random.uniform(1, 5)
        ellipses.append(EllipseParameters(center_x, center_y, radius_x, radius_y, intensity))
    return ellipses

def create_kspace_data(num_samples, num_ellipses_per_sample, kx, ky):
    """Crée plusieurs ensembles de données k-space à partir de plusieurs fantômes ellipsoïdes.

    Parameters
    ----------
    num_samples : int
        Le nombre d'ensembles de données k-space à générer.
    num_ellipses_per_sample : int
        Le nombre d'ellipses dans chaque fantôme.
    kx : torch.Tensor
        Les positions k-space dans la direction kx.
    ky : torch.Tensor
        Les positions k-space dans la direction ky.

    Returns
    -------
    List[torch.Tensor]
        Une liste de tenseurs contenant les données k-space.
    """
    kspace_data_list = []
    for _ in range(num_samples):
        ellipses = generate_random_ellipses(num_ellipses_per_sample)
        phantom = EllipsePhantom(ellipses)
        kspace_data = phantom.kspace(ky, kx)
        kspace_data_list.append(kspace_data)
    return kspace_data_list

# Exemple d'utilisation
num_samples = 10  # Nombre d'ensembles de données k-space à générer
num_ellipses_per_sample = 3  # Nombre d'ellipses dans chaque fantôme

# Génération des positions k-space
kx = torch.linspace(-128, 127, 256)
ky = torch.linspace(-128, 127, 256)
kx, ky = torch.meshgrid(kx, ky)

# Création des données k-space
kspace_data_samples = create_kspace_data(num_samples, num_ellipses_per_sample, kx, ky)

# kspace_data_samples contiendra une liste de tenseurs k-space pour chaque échantillon
for i, kspace_data in enumerate(kspace_data_samples):
    print(f"K-space data sample {i+1}: shape = {kspace_data.shape}")


K-space data sample 1: shape = torch.Size([256, 256])
K-space data sample 2: shape = torch.Size([256, 256])
K-space data sample 3: shape = torch.Size([256, 256])
K-space data sample 4: shape = torch.Size([256, 256])
K-space data sample 5: shape = torch.Size([256, 256])
K-space data sample 6: shape = torch.Size([256, 256])
K-space data sample 7: shape = torch.Size([256, 256])
K-space data sample 8: shape = torch.Size([256, 256])
K-space data sample 9: shape = torch.Size([256, 256])
K-space data sample 10: shape = torch.Size([256, 256])


/data/bouill01/conda_envs/py311/lib/python3.11/site-packages/torch/functional.py:512: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3587.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [15]:
class YourDataset(torch.utils.data.Dataset):
    def __init__(self, data_list):
        self.data_list = data_list

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        return self.data_list[idx]

In [16]:
# Define values to compare for us_idx
us_idx_values = torch.arange(0, 127, 3)[None, :]

In [17]:
# x_true fully sampled
iterative_sense_reconstruction = mrpro.algorithms.reconstruction.DirectReconstruction.from_kdata(shifted_kdatapuls)
x_true = iterative_sense_reconstruction(shifted_kdatapuls)
x_true = x_true.data

In [18]:
dcf_operator = mrpro.data.DcfData.from_traj_voronoi(shifted_kdatapuls.traj).as_operator()

# Define Fourier operator using the trajectory and header information in kdata
fourier_operator = mrpro.operators.FourierOp.from_kdata(shifted_kdatapuls)

img_coilwise = mrpro.data.IData.from_tensor_and_kheader(*fourier_operator.H(*dcf_operator(shifted_kdatapuls.data)), shifted_kdatapuls.header)
csm_operator = mrpro.data.CsmData.from_idata_walsh(img_coilwise).as_operator()

# Create the acquisition operator A
acquisition_operator = fourier_operator @ csm_operator


# Unpack the tuple and perform the subtraction
kdata_simulated = acquisition_operator(x_true)

# Extract the k-space data from the tuple
kdata_simulated = kdata_simulated[0]    

# Create a KData object with the simulated k-space data
kdata_simulated_object = mrpro.data.KData(
    data=kdata_simulated,  
    traj=shifted_kdatapuls.traj, 
    header=shifted_kdatapuls.header 
)

# Split k-space data into other dimensions based on undersampling indices
kdata_simulated_object_us = KData.split_k1_into_other(kdata_simulated_object, us_idx_values , other_label='repetition')

In [19]:
cf_operator_rad = mrpro.data.DcfData.from_traj_voronoi(kdata_simulated_object_us.traj).as_operator()
fourier_operator_rad = mrpro.operators.FourierOp.from_kdata(kdata_simulated_object_us)
img_coilwise_rad = mrpro.data.IData.from_tensor_and_kheader(*fourier_operator_rad.H(*cf_operator_rad(kdata_simulated_object_us.data)), kdata_simulated_object_us.header)
csm_operator_rad = mrpro.data.CsmData.from_idata_walsh(img_coilwise_rad).as_operator()
acquisition_operator_rad = fourier_operator_rad @ csm_operator_rad

# Initialiser les hyperparamètres
n_epochs = 10
learning_rate = 1e-4
batch_size = 16


In [20]:
# Initialiser les objets nécessaires
unet = UNet(dim=2, n_ch_in=2, n_ch_out=2, n_enc_stages=3, n_convs_per_stage=2, n_filters=16)
model = NUFFTCascade(acquisition_operator_rad, unet, nu=1, npcg=16, w=0.1, learn_lambda=True)
loss_fct = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [21]:
# Exemple de données (remplacez par vos données réelles)
data_list = [(torch.randn(2, 256, 256), torch.randn(2, 256, 256)) for _ in range(100)]
dataset = YourDataset(data_list)
data_loader = DataLoader(dataset=dataset, batch_size=batch_size, shuffle=True)

In [22]:
# Entraînement du modèle
for epoch in range(n_epochs):
    for _, data in enumerate(data_loader):
        xu, xf = data
        optimizer.zero_grad()
        xreco = model(xu, xf)
        loss = loss_fct(xreco, xf)
        loss.backward()
        optimizer.step()
        print(f'Epoch [{epoch+1}/{n_epochs}], Loss: {loss.item():.4f}')

print("Training finished.")

TypeError: unsupported operand type(s) for -: 'tuple' and 'Tensor'

In [24]:
kdata_simulated_object_us.traj.ky.shape

torch.Size([1, 1, 43, 220])

In [ ]:
xf.shape

torch.Size([16, 2, 256, 256])

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader
import numpy as np

class NUFFTCascade(nn.Module):
    def __init__(self, acquisition_operator, unet, nu, npcg, w, learn_lambda=True):
        super(NUFFTCascade, self).__init__()
        self.acquisition_operator = acquisition_operator
        self.unet = unet
        self.nu = nu
        self.npcg = npcg
        self.w = nn.Parameter(torch.tensor(w), requires_grad=True)
        self.lambda_param = nn.Parameter(torch.tensor(0.1), requires_grad=learn_lambda)

    def forward(self, x, k_space_data):
        for _ in range(self.npcg):
            # Handle the tuple returned by acquisition_operator
            op_x = self.acquisition_operator(x)[0] - k_space_data
            operator_test = self.acquisition_operator.H(op_x)
            
            grad_unet_x = torch.autograd.grad(outputs=self.unet(x), inputs=x, grad_outputs=torch.ones_like(self.unet(x)), retain_graph=True)[0]
            x = x - self.w * operator_test - self.w * grad_unet_x
        return x

class YourDataset(torch.utils.data.Dataset):
    def __init__(self, data_list):
        self.data_list = data_list

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        return self.data_list[idx]

# Define values to compare for us_idx
us_idx_values = torch.arange(0, 127, 3)[None, :]

# x_true fully sampled
iterative_sense_reconstruction = mrpro.algorithms.reconstruction.DirectReconstruction.from_kdata(shifted_kdatapuls)
x_true = iterative_sense_reconstruction(shifted_kdatapuls)
x_true = x_true.data

dcf_operator = mrpro.data.DcfData.from_traj_voronoi(shifted_kdatapuls.traj).as_operator()

# Define Fourier operator using the trajectory and header information in kdata
fourier_operator = mrpro.operators.FourierOp.from_kdata(shifted_kdatapuls)

img_coilwise = mrpro.data.IData.from_tensor_and_kheader(*fourier_operator.H(*dcf_operator(shifted_kdatapuls.data)), shifted_kdatapuls.header)
csm_operator = mrpro.data.CsmData.from_idata_walsh(img_coilwise).as_operator()

# Create the acquisition operator A
acquisition_operator = fourier_operator @ csm_operator

# Unpack the tuple and perform the subtraction
kdata_simulated = acquisition_operator(x_true)

# Extract the k-space data from the tuple
kdata_simulated = kdata_simulated[0]    

# Create a KData object with the simulated k-space data
kdata_simulated_object = mrpro.data.KData(
    data=kdata_simulated,  
    traj=shifted_kdatapuls.traj, 
    header=shifted_kdatapuls.header 
)

# Split k-space data into other dimensions based on undersampling indices
kdata_simulated_object_us = KData.split_k1_into_other(kdata_simulated_object, us_idx_values , other_label='repetition')

cf_operator_rad = mrpro.data.DcfData.from_traj_voronoi(kdata_simulated_object_us.traj).as_operator()
fourier_operator_rad = mrpro.operators.FourierOp.from_kdata(kdata_simulated_object_us)
img_coilwise_rad = mrpro.data.IData.from_tensor_and_kheader(*fourier_operator_rad.H(*cf_operator_rad(kdata_simulated_object_us.data)), kdata_simulated_object_us.header)
csm_operator_rad = mrpro.data.CsmData.from_idata_walsh(img_coilwise_rad).as_operator()
acquisition_operator_rad = fourier_operator_rad @ csm_operator_rad

# Initialiser les hyperparamètres
n_epochs = 10
learning_rate = 1e-4
batch_size = 16

# Initialiser les objets nécessaires
unet = UNet(dim=2, n_ch_in=2, n_ch_out=2, n_enc_stages=3, n_convs_per_stage=2, n_filters=16)
model = NUFFTCascade(acquisition_operator_rad, unet, nu=1, npcg=16, w=0.1, learn_lambda=True)
loss_fct = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# Exemple de données (remplacez par vos données réelles)
data_list = [(torch.randn(2, 256, 256), torch.randn(2, 256, 256)) for _ in range(100)]
dataset = YourDataset(data_list)
data_loader = DataLoader(dataset=dataset, batch_size=batch_size, shuffle=True)

# Entraînement du modèle
for epoch in range(n_epochs):
    for _, data in enumerate(data_loader):
        xu, xf = data
        optimizer.zero_grad()
        xreco = model(xu, xf)
        loss = loss_fct(xreco, xf)
        loss.backward()
        optimizer.step()
        print(f'Epoch [{epoch+1}/{n_epochs}], Loss: {loss.item():.4f}')

print("Training finished.")


RuntimeError: The size of tensor a (220) must match the size of tensor b (256) at non-singleton dimension 4

NameError: name 'k_space_data' is not defined

In [ ]:
# Define values to compare for us_idx
us_idx_values = torch.arange(0, 127, 3)[None, :]

In [ ]:
# x_true fully sampled
iterative_sense_reconstruction = mrpro.algorithms.reconstruction.DirectReconstruction.from_kdata(shifted_kdatapuls)
x_true = iterative_sense_reconstruction(shifted_kdatapuls)
x_true = x_true.data

In [ ]:
dcf_operator = mrpro.data.DcfData.from_traj_voronoi(shifted_kdatapuls.traj).as_operator()

# Define Fourier operator using the trajectory and header information in kdata
fourier_operator = mrpro.operators.FourierOp.from_kdata(shifted_kdatapuls)

img_coilwise = mrpro.data.IData.from_tensor_and_kheader(*fourier_operator.H(*dcf_operator(shifted_kdatapuls.data)), shifted_kdatapuls.header)
csm_operator = mrpro.data.CsmData.from_idata_walsh(img_coilwise).as_operator()

# Create the acquisition operator A
acquisition_operator = fourier_operator @ csm_operator


# Unpack the tuple and perform the subtraction
kdata_simulated = acquisition_operator(x_true)

# Extract the k-space data from the tuple
kdata_simulated = kdata_simulated[0]    

# Create a KData object with the simulated k-space data
kdata_simulated_object = mrpro.data.KData(
    data=kdata_simulated,  
    traj=shifted_kdatapuls.traj, 
    header=shifted_kdatapuls.header 
)

# Split k-space data into other dimensions based on undersampling indices
kdata_simulated_object_us = KData.split_k1_into_other(kdata_simulated_object, us_idx_values , other_label='repetition')

In [ ]:
dcf_operator = mrpro.data.DcfData.from_traj_voronoi(kdata_simulated_object_us.traj).as_operator()

# Define Fourier operator using the trajectory and header information in kdata
fourier_operator = mrpro.operators.FourierOp.from_kdata(kdata_simulated_object_us)

# Calculate coil maps
# Note that operators return a tuple of tensors, so we need to unpack it,
# even though there is only one tensor returned from adjoint operator.
img_coilwise = mrpro.data.IData.from_tensor_and_kheader(*fourier_operator.H(*dcf_operator(kdata_simulated_object_us.data)), kdata_simulated_object_us.header)
csm_operator = mrpro.data.CsmData.from_idata_walsh(img_coilwise).as_operator()

# Create the acquisition operator A
acquisition_operator = fourier_operator @ csm_operator

In [ ]:
# Define the UNet model (assuming the UNet class is already defined as above)
dim = 2
n_ch_in = 2
n_ch_out = 2
n_enc_stages = 3
n_convs_per_stage = 2
n_filters = 16
kernel_size = 3
res_connection = False
bias = False

unet_model = UNet(dim, n_ch_in, n_ch_out, n_enc_stages, n_convs_per_stage, n_filters, kernel_size, res_connection, bias)

In [ ]:
x_dagger = acquisition_operator.H(kdata_simulated_object_us.data)

In [ ]:
Reg = unet_model(x_dagger)

TypeError: conv2d() received an invalid combination of arguments - got (tuple, Parameter, NoneType, tuple, tuple, tuple, int), but expected one of:
 * (Tensor input, Tensor weight, Tensor bias, tuple of ints stride, tuple of ints padding, tuple of ints dilation, int groups)
      didn't match because some of the arguments have invalid types: (!tuple of (Tensor,)!, !Parameter!, !NoneType!, !tuple of (int, int)!, !tuple of (int, int)!, !tuple of (int, int)!, int)
 * (Tensor input, Tensor weight, Tensor bias, tuple of ints stride, str padding, tuple of ints dilation, int groups)
      didn't match because some of the arguments have invalid types: (!tuple of (Tensor,)!, !Parameter!, !NoneType!, !tuple of (int, int)!, !tuple of (int, int)!, !tuple of (int, int)!, int)


In [ ]:
right_hand_side_low_res_filt= acquisition_operator_result + lamda * x_cnn

In [ ]:
cg_operator_low_res_filt = acquisition_operator.H @ acquisition_operator + lambda_operator

In [ ]:
img_manual_low_res_filt = mrpro.algorithms.optimizers.cg(
    cg_operator_low_res_filt, right_hand_side_low_res_filt, initial_value=right_hand_side_low_res_filt, max_iterations=4
)